# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [ ]:
import os # Import the os module to access environment variables
from dotenv import load_dotenv # Import load_dotenv to load variables from the .env file

load_dotenv(override=True) # Load environment variables from the .env file, overriding any existing ones
api_key = os.getenv('GROQ_API_KEY') # Retrieve the GROQ_API_KEY environment variable

# Validate the presence and structure of the Groq API key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("gsk_"):
    print("An API key was found, but it doesn't start with gsk_; please check you're using the right key.")
else:
    print("API key found and looks good so far!")

# 1. Imports the standard 'os' library and the 'load_dotenv' function from 'python-dotenv' to manage local environment variables securely.
# 2. Calls 'load_dotenv(override=True)' to load environmental variables from a '.env' file, ensuring any pre-existing environment configurations are overridden.
# 3. Fetches the 'GROQ_API_KEY' variable using 'os.getenv()'.
# 4. Validates if the key is present and verify if it starts with the correct Groq API prefix 'gsk_', printing status messages accordingly.


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [ ]:
import requests # Import the requests library to send HTTP requests

# Construct authorization headers for the Groq API
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

# Define the payload with the target model and user message for Groq
payload = {
    "model": "llama-3.1-8b-instant", # The selected open-source model hosted on Groq
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"} # A list of conversation messages containing a single user prompt
    ]
}

payload # Output the payload structure

# 1. Imports the 'requests' library to interact directly with REST API endpoints over HTTP.
# 2. Prepares 'headers' with an 'Authorization' Bearer token using the retrieved 'api_key' and sets the 'Content-Type' to JSON.
# 3. Formats the JSON 'payload' dict to configure the request parameters, including the open-source 'llama-3.1-8b-instant' model and a structured 'messages' list containing the user prompt.
# 4. Outputs the raw payload dictionary structure for preview.


In [ ]:
# Execute a POST request directly to the Groq Chat Completions endpoint
response = requests.post(
    "https://api.groq.com/openai/v1/chat/completions", # The official Groq API endpoint
    headers=headers, # Authorization and content headers
    json=payload # Request payload parameters
)

response.json() # Parse and output the response payload as a JSON dictionary

# 1. Calls 'requests.post' to send a synchronous HTTP POST request to Groq's official REST endpoint at 'https://api.groq.com/openai/v1/chat/completions'.
# 2. Passes the authentication headers and the JSON payload containing the model and prompt details.
# 3. Invokes '.json()' on the response object to parse the raw HTTP response string into a readable Python dictionary structure.


In [ ]:
# Extract the actual text response from the parsed JSON response object
response.json()["choices"][0]["message"]["content"]

# 1. Navigates the nested JSON response dictionary returned by the Groq API.
# 2. Retrieves the first completion choice at list index '0' under the 'choices' key.
# 3. Accesses the 'message' dictionary and extracts the text content under the 'content' key.


# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [ ]:
# Create Groq client and call the API
from groq import Groq # Import the Groq class from the groq library
client = Groq() # Initialize the Groq client, which automatically picks up GROQ_API_KEY from environment

# Request a chat completion from Groq using the official Python SDK wrapper
response = client.chat.completions.create(
    model="llama-3.1-8b-instant", # Select the open-source Llama model
    messages=[{"role": "user", "content": "Tell me a fun fact"}] # Provide the structured prompt
)

response.choices[0].message.content # Extract and output the response text

# 1. Imports the official 'Groq' client wrapper from the 'groq' package.
# 2. Instantiates 'client = Groq()', which automatically reads the 'GROQ_API_KEY' variable from the loaded environment.
# 3. Invokes the client's 'chat.completions.create()' method, passing the 'llama-3.1-8b-instant' model and a list of message objects.
# 4. Accesses the structured 'response' object using dot notation to print the text response content from the first generation choice.


## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, Groq provides an OpenAI compatible endpoint at: `https://api.groq.com/openai/v1`

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
groq_compat = OpenAI(base_url="https://api.groq.com/openai/v1", api_key="gsk_...")
groq_compat.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!


In [ ]:
GROQ_BASE_URL = "https://api.groq.com/openai/v1" # Define Groq's OpenAI-compatible base URL

load_dotenv(override=True) # Load the environment variables from the .env file

groq_api_key = os.getenv("GROQ_API_KEY") # Retrieve the Groq API key

# Validate the presence and structure of the Groq API key
if not groq_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file!")
elif not groq_api_key.startswith("gsk_"):
    print("An API key was found, but it doesn't start with gsk_")
else:
    print("API key found and looks good so far!")

# 1. Defines the 'GROQ_BASE_URL' pointing to Groq's OpenAI-compatible REST endpoint.
# 2. Loads environmental variables using 'load_dotenv(override=True)'.
# 3. Pulls the 'GROQ_API_KEY' variable and validates that it is present and correctly prefixed with 'gsk_'.


In [ ]:
from openai import OpenAI # Import the OpenAI client library class
# Initialize the OpenAI client using Groq's compatible base URL and key
groq_compat = OpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

# Use the OpenAI client instance to call Groq's llama model
response = groq_compat.chat.completions.create(
    model="llama-3.1-8b-instant", # The open-source model hosted on Groq
    messages=[{"role": "user", "content": "Tell me a fun fact"}] # The structured user prompt
)

response.choices[0].message.content # Extract and output the response text

# 1. Imports the standard 'OpenAI' client class from the 'openai' package.
# 2. Instantiates 'OpenAI' but customizes the connection by passing 'base_url=GROQ_BASE_URL' and 'api_key=groq_api_key'. This overrides the target endpoint to route API calls directly to Groq instead of OpenAI.
# 3. Invokes 'chat.completions.create()' identical to an OpenAI call, targeting the 'llama-3.1-8b-instant' model.
# 4. Parses the returned OpenAI-compatible response object to output the generated text choice.


## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [ ]:
requests.get("http://localhost:11434").content # Call local Ollama status endpoint and return response content

# 1. Uses the 'requests' library to send a GET request to 'http://localhost:11434', where the local Ollama server runs.
# 2. Accesses the raw '.content' attribute of the response, which should return the text 'Ollama is running' if the service is active.


### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [ ]:
!ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1" # Define local Ollama's OpenAI-compatible base URL

# Initialize the OpenAI client using Ollama's local base URL and a mock key
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# 1. Sets 'OLLAMA_BASE_URL' to target the local Ollama server's OpenAI-compatible endpoint route ('/v1').
# 2. Instantiates the 'OpenAI' client, passing the local base URL and a dummy 'api_key' string ('ollama'), which Ollama requires for standard compliance but ignores during authentication.


In [ ]:
# Request a chat completion from the local Ollama server
response = ollama.chat.completions.create(
    model="llama3.2", # The local Llama model pulled onto your machine
    messages=[{"role": "user", "content": "Tell me a fun fact"}]
)

response.choices[0].message.content # Extract and output the response text

# 1. Calls the 'chat.completions.create()' method on the local 'ollama' client instance.
# 2. Configures the query for the locally pulled model 'llama3.2' and sends the user message list.
# 3. Accesses the standard response format to extract and print the text response content from the first generation.


In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
# Request a chat completion from the local Ollama server using DeepSeek
response = ollama.chat.completions.create(
    model="deepseek-r1:1.5b", # The distilled deepseek-r1 local model
    messages=[{"role": "user", "content": "Tell me a fun fact"}]
)

response.choices[0].message.content # Extract and output the response text

# 1. Calls the 'chat.completions.create()' method on the local 'ollama' client instance.
# 2. Targets the local 'deepseek-r1:1.5b' model (which uses the chain-of-thought thinking process).
# 3. Extracts and prints the text content generated by the local deepseek model.


# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`